# 10A — Feature Interaction Screening for Road Accident Severity

## Objective

This notebook screens **candidate feature interactions** before creating engineered interaction features for the severity prediction models.

The workflow is deliberately separated into two stages:

1. **Interaction screening:** identify combinations of existing, leakage-safe features whose severity composition varies meaningfully across their combinations.
2. **Feature engineering:** use the shortlisted interactions in the next notebook and test whether they improve downstream model performance.

### Leakage control

- The target `SEVERITY` is used only for screening/evaluation.
- Candidate interactions are generated from the **26 original features selected by Information Gain**.
- Screening is performed using the **training set only**.
- Outcome-derived columns removed during preprocessing are not used.

This means the test set remains untouched during interaction discovery.


In [1]:
# Imports and configuration
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_DIR = "../data"

print("Ready.")


Ready.


## 1. Load the preprocessed training data

Notebook 03 produced the final encoded train/test matrices. Notebook 04 produced the Information Gain-selected original feature list.

The selected feature file uses the column name `original_feature`.


In [2]:
# Load preprocessed matrices and metadata
X_train = sparse.load_npz(os.path.join(DATA_DIR, "X_train_final.npz")).tocsr()
y_train = np.load(os.path.join(DATA_DIR, "y_train_encoded.npy"))

feature_names = pd.read_csv(
    os.path.join(DATA_DIR, "final_feature_names.csv")
)["feature_name"].tolist()

selected_df = pd.read_csv(
    os.path.join(DATA_DIR, "selected_features_ig.csv")
)
selected_original = selected_df["original_feature"].dropna().tolist()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("Encoded feature count:", len(feature_names))
print("IG-selected original features:", len(selected_original))
print("\nSelected original features:")
print(selected_original)


X_train shape: (160278, 361)
y_train shape: (160278,)
Encoded feature count: 361
IG-selected original features: 26

Selected original features:
['POLICE_ATTEND', 'DCA_CODE', 'DRIVER', 'LONGITUDE', 'ROAD_ROUTE_1', 'PASSENGERVEHICLE', 'NO_OF_VEHICLES', 'VICGRID_Y', 'LATITUDE', 'VICGRID_X', 'MOTORCYCLIST', 'DCA_CODE_DESCRIPTION', 'ACCIDENT_TYPE', 'PEDESTRIAN', 'YOUNG_DRIVER_18_25', 'RUN_OFFROAD', 'DEG_URBAN_NAME', 'MOTORCYCLE', 'SPEED_ZONE', 'OLD_DRIVER_75_AND_OVER', 'TOTAL_PERSONS', 'OLD_PED_65_AND_OVER', 'FEMALES', 'BICYCLIST', 'MALES', 'ROAD_NAME_FREQ']


## 2. Decode the selected original features back to interpretable variables

The preprocessing stage expanded categorical variables into one-hot columns. For interaction screening, however, we want to reason at the **original-feature level**.

Examples:

- `SPEED_ZONE` → the original speed-zone variable
- `MOTORCYCLE` → motorcycle involvement
- `DEG_URBAN_NAME` → urban/rural context
- `ACCIDENT_TYPE` → crash type

For categorical variables, the screening below uses the original categorical values. For numerical variables, values are discretised into a small number of quantile-based bins where appropriate.

This is a screening tool, not the final engineered representation.


In [3]:
# Build an interpretable screening dataframe from the already-saved preprocessed matrix.
# The original CSV is intentionally NOT required here. This keeps the notebook portable
# and guarantees that interaction screening uses exactly the same training rows/features
# as Notebook 03/04.

from scipy import sparse

# Load the encoded training matrix and target created by Notebook 03.
X_train_screen = sparse.load_npz(
    os.path.join(DATA_DIR, "X_train_final.npz")
).tocsr()
y_train_screen = np.load(
    os.path.join(DATA_DIR, "y_train_encoded.npy")
)

# Decode the selected 136 encoded columns back to one interpretable value per
# selected original feature.
selected_encoded_names = []
selected_groups = {}

for original_feature in selected_original:
    # Exact match handles numerical features.
    exact = [i for i, name in enumerate(feature_names) if name == original_feature]

    # Prefix matching is used ONLY for one-hot encoded categorical features.
    # The underscore prevents collisions such as DCA_CODE vs DCA_CODE_DESCRIPTION.
    prefixed = [
        i for i, name in enumerate(feature_names)
        if name.startswith(original_feature + "_")
    ]

    matches = exact if exact else prefixed

    if matches:
        selected_groups[original_feature] = matches
        selected_encoded_names.extend(matches)

selected_encoded_names = list(dict.fromkeys(selected_encoded_names))

print("Encoded training matrix:", X_train_screen.shape)
print("Selected original features:", len(selected_groups))
print("Selected encoded columns:", len(selected_encoded_names))

# Create the interpretable dataframe.
decoded = {}

for original_feature, indices in selected_groups.items():
    block = X_train_screen[:, indices]

    if len(indices) == 1:
        # Numerical / frequency-encoded feature.
        decoded[original_feature] = np.asarray(block.toarray()).ravel()
    else:
        # One-hot categorical feature: recover the active category.
        dense_block = block.toarray()
        names = [feature_names[i] for i in indices]

        values = np.array([
            names[row.argmax()] if row.max() > 0 else "Missing"
            for row in dense_block
        ], dtype=object)

        # Remove the original feature prefix from one-hot labels for readability.
        prefix = original_feature + "_"
        values = np.array([
            v[len(prefix):] if isinstance(v, str) and v.startswith(prefix) else v
            for v in values
        ], dtype=object)

        decoded[original_feature] = values

screen_df = pd.DataFrame(decoded)
screen_df["SEVERITY"] = y_train_screen

# Convert encoded target labels to the project labels used elsewhere.
label_map = {0: "Fatal", 1: "Other Injury", 2: "Serious Injury"}
screen_df["SEVERITY"] = screen_df["SEVERITY"].map(label_map)

print("Screening dataframe:", screen_df.shape)
print("Severity distribution:")
print(screen_df["SEVERITY"].value_counts())


Encoded training matrix: (160278, 361)
Selected original features: 26
Selected encoded columns: 136
Screening dataframe: (160278, 27)
Severity distribution:
SEVERITY
Other Injury      99953
Serious Injury    57643
Fatal              2682
Name: count, dtype: int64


### Important note about the screening data

The interaction screen works directly from the saved **training matrix produced by Notebook 03**. Categorical one-hot columns are decoded back to their original feature values, while numerical/frequency-encoded features are retained for screening. This ensures the screen uses exactly the same training population and preprocessing decisions as the earlier notebooks.

No test-set information is used to choose interactions.


In [4]:
# Confirm that all selected original features exist in the raw training data.
available_selected = [f for f in selected_original if f in screen_df.columns]
missing_selected = [f for f in selected_original if f not in screen_df.columns]

print("Available selected original features:", len(available_selected))
print("Missing from raw data:", missing_selected)


Available selected original features: 26
Missing from raw data: []


## 3. Define candidate feature families

Candidate interactions are generated from different conceptual groups rather than taking every possible pair.

This avoids creating a large number of arbitrary combinations and follows the project logic that crash severity can depend on interacting road, vehicle, human, and environmental factors.

### Candidate families

- **Crash:** accident type, DCA information, run-off-road, number of vehicles
- **Road/environment:** speed zone, urban/rural context, road geometry
- **Road users:** motorcycle, motorcyclist, pedestrian, bicyclist, passenger vehicle, heavy vehicle
- **Driver age:** young/older driver indicators
- **Pedestrian age:** older pedestrian indicator
- **Context:** police attendance and time-related information

Only features actually present in the IG-selected list are considered.


In [5]:
# Candidate feature families
families = {
    "crash": [
        "ACCIDENT_TYPE", "DCA_CODE", "DCA_CODE_DESCRIPTION",
        "NO_OF_VEHICLES", "RUN_OFFROAD"
    ],
    "road_environment": [
        "SPEED_ZONE", "DEG_URBAN_NAME", "ROAD_ROUTE_1",
        "ROAD_NAME_FREQ"
    ],
    "road_users": [
        "MOTORCYCLE", "MOTORCYCLIST", "PEDESTRIAN",
        "BICYCLIST", "PASSENGERVEHICLE", "DRIVER",
        "HEAVYVEHICLE"
    ],
    "driver_age": [
        "YOUNG_DRIVER_18_25", "OLD_DRIVER_75_AND_OVER"
    ],
    "pedestrian_age": [
        "OLD_PED_65_AND_OVER"
    ],
    "context": [
        "POLICE_ATTEND", "HOUR"
    ]
}

# Restrict to the IG-selected features.
for group in families:
    families[group] = [f for f in families[group] if f in available_selected]

for group, values in families.items():
    print(f"{group}: {values}")


crash: ['ACCIDENT_TYPE', 'DCA_CODE', 'DCA_CODE_DESCRIPTION', 'NO_OF_VEHICLES', 'RUN_OFFROAD']
road_environment: ['SPEED_ZONE', 'DEG_URBAN_NAME', 'ROAD_ROUTE_1', 'ROAD_NAME_FREQ']
road_users: ['MOTORCYCLE', 'MOTORCYCLIST', 'PEDESTRIAN', 'BICYCLIST', 'PASSENGERVEHICLE', 'DRIVER']
driver_age: ['YOUNG_DRIVER_18_25', 'OLD_DRIVER_75_AND_OVER']
pedestrian_age: ['OLD_PED_65_AND_OVER']
context: ['POLICE_ATTEND']


## 4. Generate two-way candidate interactions

The first screen focuses on interactions between **different feature families**.

For example:

`MOTORCYCLE × SPEED_ZONE`

is considered because it combines a road-user characteristic with a road/environment characteristic.

We deliberately avoid same-family combinations at this stage because they are more likely to be redundant.


In [6]:
from itertools import combinations

candidate_pairs = []

family_names = list(families.keys())

for i in range(len(family_names)):
    for j in range(i + 1, len(family_names)):
        g1, g2 = family_names[i], family_names[j]
        for f1 in families[g1]:
            for f2 in families[g2]:
                if f1 != f2:
                    candidate_pairs.append((f1, f2, g1, g2))

candidate_pairs_df = pd.DataFrame(
    candidate_pairs,
    columns=["feature_a", "feature_b", "family_a", "family_b"]
).drop_duplicates()

print("Candidate two-way interactions:", len(candidate_pairs_df))
candidate_pairs_df.head(20)


Candidate two-way interactions: 139


,feature_a,feature_b,family_a,family_b
0,ACCIDENT_TYPE,SPEED_ZONE,crash,road_environment
1,ACCIDENT_TYPE,DEG_URBAN_NAME,crash,road_environment
2,ACCIDENT_TYPE,ROAD_ROUTE_1,crash,road_environment
3,ACCIDENT_TYPE,ROAD_NAME_FREQ,crash,road_environment
4,DCA_CODE,SPEED_ZONE,crash,road_environment
5,DCA_CODE,DEG_URBAN_NAME,crash,road_environment
6,DCA_CODE,ROAD_ROUTE_1,crash,road_environment
7,DCA_CODE,ROAD_NAME_FREQ,crash,road_environment
8,DCA_CODE_DESCRIPTION,SPEED_ZONE,crash,road_environment
9,DCA_CODE_DESCRIPTION,DEG_URBAN_NAME,crash,road_environment


## 5. Prepare interpretable screening variables

Categorical variables are kept as categories.

Continuous/count variables are converted into a small number of quantile bins. This prevents a continuous variable such as `NO_OF_VEHICLES` from producing hundreds of tiny interaction groups.

The binning is learned from the training data only.


In [7]:
# Identify binary/count/numeric features that should be binned for screening.
numeric_features = screen_df[available_selected].select_dtypes(include=np.number).columns.tolist()

screen_features = {}

for f in available_selected:
    s = screen_df[f].copy()

    # Treat low-cardinality numeric variables as categorical.
    nunique = s.nunique(dropna=True)

    if pd.api.types.is_numeric_dtype(s) and nunique > 8:
        # Quantile bins; duplicates are dropped if the distribution is discrete.
        try:
            binned = pd.qcut(s, q=4, duplicates="drop")
            screen_features[f] = binned.astype(str)
        except Exception:
            screen_features[f] = s.astype(str)
    else:
        screen_features[f] = s.fillna("Missing").astype(str)

screen_features_df = pd.DataFrame(screen_features, index=screen_df.index)

print("Screening matrix shape:", screen_features_df.shape)
print(screen_features_df.head())


Screening matrix shape: (160278, 26)
  POLICE_ATTEND         DCA_CODE            DRIVER         LONGITUDE  \
0            No  (-0.334, 1.093]  (-1.684, -0.554]  (-5.013, -0.169]   
1           Yes  (-0.334, 1.093]  (-1.684, -0.554]  (-5.013, -0.169]   
2           Yes   (1.093, 2.326]  (-1.684, -0.554]  (-0.169, 0.0534]   
3           Yes   (1.093, 2.326]  (-1.684, -0.554]  (-5.013, -0.169]   
4            No  (-0.72, -0.334]  (-1.684, -0.554]  (-0.169, 0.0534]   

       ROAD_ROUTE_1 PASSENGERVEHICLE   NO_OF_VEHICLES         VICGRID_Y  \
0  (-2.179, -0.525]  (-1.591, -0.49]  (-1.111, 0.234]   (0.0413, 6.456]   
1   (-0.242, 1.159]  (0.609, 19.297]  (0.234, 25.783]   (0.0413, 6.456]   
2  (-0.525, -0.242]  (-1.591, -0.49]  (-1.111, 0.234]  (-0.434, -0.178]   
3  (-2.179, -0.525]  (-1.591, -0.49]  (-1.111, 0.234]  (-2.391, -0.434]   
4  (-2.179, -0.525]  (-1.591, -0.49]  (-1.111, 0.234]  (-2.391, -0.434]   

           LATITUDE         VICGRID_X  ...           DEG_URBAN_NAME  \
0   (0.0

## 6. Score an interaction by severity heterogeneity

For each candidate pair we create a combined category:

`feature_a = value_a | feature_b = value_b`

Then we calculate:

- number of observed combinations,
- minimum group size,
- weighted severity entropy,
- variation in fatal/serious-injury proportions across sufficiently large groups.

The main ranking signal is **severity composition variability**.

A useful interaction should not merely have many categories; its combinations should show meaningfully different severity distributions.

Small groups are ignored using a minimum-support threshold to reduce unstable conclusions.


In [8]:
from sklearn.metrics import mutual_info_score

MIN_SUPPORT = 100

# Global training severity proportions
severity_order = ["Fatal", "Other Injury", "Serious Injury"]

def interaction_screen_score(a, b, y):
    tmp = pd.DataFrame({
        "a": a.astype(str),
        "b": b.astype(str),
        "severity": y.astype(str)
    })
    tmp["interaction"] = tmp["a"] + " | " + tmp["b"]

    counts = tmp["interaction"].value_counts()
    valid = counts[counts >= MIN_SUPPORT].index

    tmp = tmp[tmp["interaction"].isin(valid)]

    if tmp.empty:
        return None

    ct = pd.crosstab(tmp["interaction"], tmp["severity"])
    ct = ct.reindex(columns=severity_order, fill_value=0)

    props = ct.div(ct.sum(axis=1), axis=0)

    # Weighted mean entropy: lower means groups are more concentrated,
    # which can indicate stronger separation by the interaction.
    p = props.clip(lower=1e-12)
    entropy = -(p * np.log2(p)).sum(axis=1)

    weights = ct.sum(axis=1) / ct.values.sum()
    weighted_entropy = float((entropy * weights).sum())

    # Variation in class proportions across sufficiently supported groups.
    fatal_range = float(props["Fatal"].max() - props["Fatal"].min())
    serious_range = float(props["Serious Injury"].max() - props["Serious Injury"].min())
    other_range = float(props["Other Injury"].max() - props["Other Injury"].min())

    # Mutual information between interaction category and severity.
    mi = float(mutual_info_score(tmp["interaction"], tmp["severity"]))

    return {
        "n_valid_combinations": len(ct),
        "min_group_support": int(ct.sum(axis=1).min()),
        "weighted_entropy": weighted_entropy,
        "fatal_range": fatal_range,
        "serious_range": serious_range,
        "other_range": other_range,
        "mutual_information": mi
    }

results = []

for _, row in candidate_pairs_df.iterrows():
    a, b = row["feature_a"], row["feature_b"]

    score = interaction_screen_score(
        screen_features_df[a],
        screen_features_df[b],
        screen_df["SEVERITY"]
    )

    if score is not None:
        results.append({
            "feature_a": a,
            "feature_b": b,
            "family_a": row["family_a"],
            "family_b": row["family_b"],
            **score
        })

interaction_results = pd.DataFrame(results)

interaction_results = interaction_results.sort_values(
    ["mutual_information", "fatal_range", "serious_range"],
    ascending=False
).reset_index(drop=True)

print("Screened interactions:", len(interaction_results))
interaction_results.head(20)


Screened interactions: 139


,feature_a,feature_b,family_a,family_b,n_valid_combinations,min_group_support,weighted_entropy,fatal_range,serious_range,other_range,mutual_information
0,DCA_CODE_DESCRIPTION,POLICE_ATTEND,crash,context,123,102,0.966618,0.118852,0.501038,0.605291,0.061809
1,PASSENGERVEHICLE,POLICE_ATTEND,road_users,context,8,254,0.975913,0.032444,0.409254,0.441698,0.054319
2,DRIVER,POLICE_ATTEND,road_users,context,8,233,0.976451,0.030146,0.414208,0.444354,0.053952
3,ACCIDENT_TYPE,POLICE_ATTEND,crash,context,17,428,0.977014,0.043984,0.419046,0.458121,0.053544
4,DCA_CODE,POLICE_ATTEND,crash,context,12,101,0.978572,0.035892,0.406046,0.441866,0.052418
5,DEG_URBAN_NAME,POLICE_ATTEND,road_environment,context,17,217,0.981575,0.054838,0.416342,0.419355,0.050461
6,MOTORCYCLIST,POLICE_ATTEND,road_users,context,6,279,0.983405,0.067857,0.500152,0.567681,0.049001
7,MOTORCYCLE,POLICE_ATTEND,road_users,context,6,282,0.983449,0.067133,0.504108,0.570912,0.048971
8,SPEED_ZONE,POLICE_ATTEND,road_environment,context,25,121,0.985228,0.058399,0.376487,0.433652,0.048072
9,RUN_OFFROAD,POLICE_ATTEND,crash,context,5,537,0.988141,0.040919,0.355653,0.396573,0.045808


## 7. Candidate interaction ranking

`Mutual information` is used as the primary screening statistic because it measures how much information the interaction category contains about severity without assuming a linear relationship.

The fatal/serious/other ranges are retained as interpretability diagnostics.

**Important:** a high screening score does not automatically mean the engineered feature will improve prediction. That is tested in the next stage.


In [9]:
# Display top candidate interactions
top_interactions = interaction_results.head(20).copy()

display(
    top_interactions[
        [
            "feature_a", "feature_b",
            "mutual_information",
            "fatal_range",
            "serious_range",
            "other_range",
            "n_valid_combinations",
            "min_group_support"
        ]
    ]
)


,feature_a,feature_b,mutual_information,fatal_range,serious_range,other_range,n_valid_combinations,min_group_support
0,DCA_CODE_DESCRIPTION,POLICE_ATTEND,0.061809,0.118852,0.501038,0.605291,123,102
1,PASSENGERVEHICLE,POLICE_ATTEND,0.054319,0.032444,0.409254,0.441698,8,254
2,DRIVER,POLICE_ATTEND,0.053952,0.030146,0.414208,0.444354,8,233
3,ACCIDENT_TYPE,POLICE_ATTEND,0.053544,0.043984,0.419046,0.458121,17,428
4,DCA_CODE,POLICE_ATTEND,0.052418,0.035892,0.406046,0.441866,12,101
5,DEG_URBAN_NAME,POLICE_ATTEND,0.050461,0.054838,0.416342,0.419355,17,217
6,MOTORCYCLIST,POLICE_ATTEND,0.049001,0.067857,0.500152,0.567681,6,279
7,MOTORCYCLE,POLICE_ATTEND,0.048971,0.067133,0.504108,0.570912,6,282
8,SPEED_ZONE,POLICE_ATTEND,0.048072,0.058399,0.376487,0.433652,25,121
9,RUN_OFFROAD,POLICE_ATTEND,0.045808,0.040919,0.355653,0.396573,5,537


## 8. Inspect severity composition of the strongest interactions

The next cell expands the strongest candidate interactions so we can see **why** they rank highly.

This is important: we are not selecting interactions from a black-box score alone. We inspect whether the high score corresponds to interpretable differences in crash severity composition.


In [10]:
def show_interaction_profile(feature_a, feature_b, min_support=MIN_SUPPORT, top_n=15):
    tmp = pd.DataFrame({
        feature_a: screen_features_df[feature_a].astype(str),
        feature_b: screen_features_df[feature_b].astype(str),
        "SEVERITY": screen_df["SEVERITY"].astype(str)
    })
    tmp["interaction"] = (
        tmp[feature_a] + " × " + tmp[feature_b]
    )

    counts = tmp["interaction"].value_counts()
    valid = counts[counts >= min_support].index

    tmp = tmp[tmp["interaction"].isin(valid)]

    profile = pd.crosstab(tmp["interaction"], tmp["SEVERITY"])
    profile = profile.reindex(columns=severity_order, fill_value=0)

    proportions = profile.div(profile.sum(axis=1), axis=0) * 100
    proportions["Support"] = profile.sum(axis=1)

    return proportions.sort_values(
        ["Fatal", "Serious Injury"],
        ascending=False
    ).head(top_n)

# Inspect top 5 screened interactions
for _, row in interaction_results.head(5).iterrows():
    print(f"\n### {row['feature_a']} × {row['feature_b']}")
    display(show_interaction_profile(row["feature_a"], row["feature_b"]))



### DCA_CODE_DESCRIPTION × POLICE_ATTEND


SEVERITY,Fatal,Other Injury,Serious Injury,Support
interaction,,,,
PED WALKING WITH TRAFFIC × Yes,11.885246,34.016393,54.098361,244
PED WALKING AGAINST TRAFFIC. × Yes,9.923664,40.458015,49.618321,131
HEAD ON (NOT OVERTAKING) × Yes,8.297124,40.099317,51.603559,4833
HEAD ON(OVERTAKING) × Yes,7.913669,41.726619,50.359712,278
PED PLAYING/LYING/WORKING/STANDING ON CARRIAGEWAY. × Yes,6.613226,53.507014,39.879760,499
OFF LEFT BEND INTO OBJECT/PARKED VEHICLE × Yes,5.978055,43.170639,50.851305,2643
OFF RIGHT BEND INTO OBJECT/PARKED VEHICLE × Yes,5.551064,45.594179,48.854756,3711
RIGHT OFF CARRIAGEWAY INTO OBJECT/PARKED VEHICLE × Yes,4.878049,43.814704,51.307247,5699
OTHER OVERTAKING MANOEUVRES NOT INCLUDED IN DCAs 150-154 × Yes,4.587156,46.788991,48.623853,109



### PASSENGERVEHICLE × POLICE_ATTEND


SEVERITY,Fatal,Other Injury,Serious Injury,Support
interaction,,,,
"(-1.591, -0.49] × Yes",3.244376,47.528742,49.226882,62539
"(-0.49, 0.609] × Yes",1.135397,63.140959,35.723643,43509
"(0.609, 19.297] × Yes",1.126574,60.818423,38.055003,12072
"(-1.591, -0.49] × Not known",0.787402,70.866142,28.346457,254
"(-1.591, -0.49] × No",0.083735,78.517898,21.398367,23885
"(0.609, 19.297] × No",0.050352,89.979859,9.969789,1986
"(-0.49, 0.609] × Not known",0.000000,89.247312,10.752688,279
"(-0.49, 0.609] × No",0.000000,91.698498,8.301502,15708



### DRIVER × POLICE_ATTEND


SEVERITY,Fatal,Other Injury,Serious Injury,Support
interaction,,,,
"(-1.684, -0.554] × Yes",3.014615,47.414855,49.570530,60074
"(-0.554, 0.576] × Yes",1.483705,62.410643,36.105652,46303
"(0.576, 22.037] × Yes",1.371030,60.938431,37.690539,11743
"(-1.684, -0.554] × Not known",0.858369,69.098712,30.042918,233
"(-1.684, -0.554] × No",0.088554,77.679876,22.231570,22585
"(0.576, 22.037] × No",0.047393,89.715640,10.236967,2110
"(-0.554, 0.576] × Not known",0.000000,89.189189,10.810811,296
"(-0.554, 0.576] × No",0.000000,91.850272,8.149728,16884



### ACCIDENT_TYPE × POLICE_ATTEND


SEVERITY,Fatal,Other Injury,Serious Injury,Support
interaction,,,,
Collision with a fixed object × Yes,4.398418,45.580638,50.020945,21485
Struck Pedestrian × Yes,3.918417,43.333668,52.747915,9953
Fall from or in moving vehicle × Yes,2.448980,47.755102,49.795918,490
Vehicle overturned (no collision) × Yes,1.849490,47.704082,50.446429,4704
Collision with vehicle × Yes,1.520744,59.332490,39.146766,76673
collision with some other object × Yes,1.515152,54.112554,44.372294,924
Struck animal × Yes,1.240951,54.601861,44.157187,967
No collision and no object struck × Yes,1.122807,49.052632,49.824561,2850
Fall from or in moving vehicle × No,0.902527,69.855596,29.241877,554



### DCA_CODE × POLICE_ATTEND


SEVERITY,Fatal,Other Injury,Serious Injury,Support
interaction,,,,
"(1.093, 2.326] × Yes",3.589203,46.504730,49.906068,30341
"(-1.492, -0.72] × Yes",2.964030,52.663106,44.372863,35391
"(-1.492, -0.72] × Not known",1.941748,74.757282,23.300971,103
"(-0.334, 1.093] × Yes",1.274763,58.428079,40.297158,23220
"(-0.72, -0.334] × Yes",0.771393,62.476001,36.752606,29168
"(1.093, 2.326] × No",0.149541,71.053194,28.797266,9362
"(-1.492, -0.72] × No",0.064960,85.215019,14.720021,7697
"(-0.334, 1.093] × No",0.009379,85.968861,14.021760,10662
"(-0.72, -0.334] × No",0.007216,90.691297,9.301487,13858


## 9. Pairwise interaction strength versus individual-feature information

An interaction should ideally add information beyond simply having two strong individual variables.

For each pair we therefore compare:

- MI(interaction, severity)
- MI(feature A, severity)
- MI(feature B, severity)

The incremental value is:

`MI(interaction) − max(MI(A), MI(B))`

A positive value means the combined representation contains additional information beyond the stronger individual feature under this screening measure.


In [11]:
# Individual-feature MI
individual_mi = {}

for f in available_selected:
    s = screen_features_df[f].astype(str)
    individual_mi[f] = mutual_info_score(s, screen_df["SEVERITY"].astype(str))

interaction_results["mi_a"] = interaction_results["feature_a"].map(individual_mi)
interaction_results["mi_b"] = interaction_results["feature_b"].map(individual_mi)

interaction_results["incremental_mi"] = (
    interaction_results["mutual_information"]
    - interaction_results[["mi_a", "mi_b"]].max(axis=1)
)

ranked_incremental = interaction_results.sort_values(
    ["incremental_mi", "mutual_information"],
    ascending=False
).reset_index(drop=True)

display(
    ranked_incremental.head(20)[
        [
            "feature_a", "feature_b",
            "mi_a", "mi_b",
            "mutual_information",
            "incremental_mi",
            "fatal_range",
            "serious_range",
            "n_valid_combinations"
        ]
    ]
)


,feature_a,feature_b,mi_a,mi_b,mutual_information,incremental_mi,fatal_range,serious_range,n_valid_combinations
0,DCA_CODE_DESCRIPTION,POLICE_ATTEND,0.029170,0.040504,0.061809,0.021305,0.118852,0.501038,123
1,ACCIDENT_TYPE,SPEED_ZONE,0.013461,0.013229,0.027637,0.014175,0.250000,0.425674,60
2,DCA_CODE_DESCRIPTION,SPEED_ZONE,0.029170,0.013229,0.043215,0.014045,0.186603,0.578442,240
3,SPEED_ZONE,DRIVER,0.013229,0.010742,0.027106,0.013877,0.065949,0.380930,29
4,PASSENGERVEHICLE,POLICE_ATTEND,0.011082,0.040504,0.054319,0.013815,0.032444,0.409254,8
5,DCA_CODE,SPEED_ZONE,0.013839,0.013229,0.027510,0.013672,0.233884,0.390515,40
6,DRIVER,POLICE_ATTEND,0.010742,0.040504,0.053952,0.013448,0.030146,0.414208,8
7,ACCIDENT_TYPE,POLICE_ATTEND,0.013461,0.040504,0.053544,0.013040,0.043984,0.419046,17
8,SPEED_ZONE,PASSENGERVEHICLE,0.013229,0.011082,0.025529,0.012300,0.053334,0.361670,28
9,DCA_CODE,POLICE_ATTEND,0.013839,0.040504,0.052418,0.011914,0.035892,0.406046,12


## 10. Select a conservative two-way shortlist

We use several safeguards rather than selecting every high-scoring pair:

- sufficient support,
- positive incremental information,
- interpretable feature pairing,
- meaningful variation in severity composition,
- no direct outcome-derived variables.

The shortlist is intentionally small enough to evaluate properly in the next notebook.


In [12]:
# Conservative shortlist.
# Keep interactions with positive incremental MI and adequate support.
shortlist = ranked_incremental[
    (ranked_incremental["incremental_mi"] > 0) &
    (ranked_incremental["min_group_support"] >= MIN_SUPPORT)
].copy()

# Limit the number carried forward so feature engineering remains controlled.
SHORTLIST_SIZE = min(12, len(shortlist))
shortlist = shortlist.head(SHORTLIST_SIZE).copy()

print(f"Two-way shortlist size: {len(shortlist)}")
display(
    shortlist[
        [
            "feature_a", "feature_b",
            "mutual_information",
            "incremental_mi",
            "fatal_range",
            "serious_range",
            "n_valid_combinations"
        ]
    ]
)


Two-way shortlist size: 12


,feature_a,feature_b,mutual_information,incremental_mi,fatal_range,serious_range,n_valid_combinations
0,DCA_CODE_DESCRIPTION,POLICE_ATTEND,0.061809,0.021305,0.118852,0.501038,123
1,ACCIDENT_TYPE,SPEED_ZONE,0.027637,0.014175,0.250000,0.425674,60
2,DCA_CODE_DESCRIPTION,SPEED_ZONE,0.043215,0.014045,0.186603,0.578442,240
3,SPEED_ZONE,DRIVER,0.027106,0.013877,0.065949,0.380930,29
4,PASSENGERVEHICLE,POLICE_ATTEND,0.054319,0.013815,0.032444,0.409254,8
5,DCA_CODE,SPEED_ZONE,0.027510,0.013672,0.233884,0.390515,40
6,DRIVER,POLICE_ATTEND,0.053952,0.013448,0.030146,0.414208,8
7,ACCIDENT_TYPE,POLICE_ATTEND,0.053544,0.013040,0.043984,0.419046,17
8,SPEED_ZONE,PASSENGERVEHICLE,0.025529,0.012300,0.053334,0.361670,28
9,DCA_CODE,POLICE_ATTEND,0.052418,0.011914,0.035892,0.406046,12


## 11. Explore a small number of three-way interactions

Three-way interactions can capture a more specific crash context, but they can also create many sparse combinations.

Therefore, they are **not exhaustively generated**.

Instead, we take the strongest two-way candidates and combine them with a third feature from a different family.

Only combinations with adequate support are retained.

Examples that may emerge include:

- motorcycle × speed zone × urban/rural context
- pedestrian × urban/rural context × hour
- heavy vehicle × speed zone × road geometry

These are candidates only; the data determines which ones survive.


In [13]:
# Build a small, controlled set of third-feature candidates.
third_feature_pool = [
    f for f in [
        "SPEED_ZONE", "DEG_URBAN_NAME", "ACCIDENT_TYPE",
        "MOTORCYCLE", "MOTORCYCLIST", "PEDESTRIAN",
        "BICYCLIST", "HEAVYVEHICLE", "NO_OF_VEHICLES",
        "RUN_OFFROAD", "HOUR",
        "YOUNG_DRIVER_18_25", "OLD_DRIVER_75_AND_OVER",
        "OLD_PED_65_AND_OVER", "ROAD_ROUTE_1"
    ]
    if f in available_selected
]

pair_lookup = {
    tuple(sorted([r["feature_a"], r["feature_b"]]))
    for _, r in shortlist.iterrows()
}

triple_candidates = []

for _, row in shortlist.head(8).iterrows():
    a, b = row["feature_a"], row["feature_b"]

    for c in third_feature_pool:
        if c in (a, b):
            continue

        triple_candidates.append((a, b, c))

# Remove duplicate unordered triples
unique_triples = []
seen = set()

for triple in triple_candidates:
    key = tuple(sorted(triple))
    if key not in seen:
        seen.add(key)
        unique_triples.append(triple)

print("Controlled three-way candidates:", len(unique_triples))
print(unique_triples[:30])


Controlled three-way candidates: 98
[('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'SPEED_ZONE'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'DEG_URBAN_NAME'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'ACCIDENT_TYPE'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'MOTORCYCLE'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'MOTORCYCLIST'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'PEDESTRIAN'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'BICYCLIST'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'NO_OF_VEHICLES'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'RUN_OFFROAD'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'YOUNG_DRIVER_18_25'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'OLD_DRIVER_75_AND_OVER'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'OLD_PED_65_AND_OVER'), ('DCA_CODE_DESCRIPTION', 'POLICE_ATTEND', 'ROAD_ROUTE_1'), ('ACCIDENT_TYPE', 'SPEED_ZONE', 'DEG_URBAN_NAME'), ('ACCIDENT_TYPE', 'SPEED_ZONE', 'MOTORCYCLE'), ('ACCIDENT_TYPE', 'SPEED_ZONE', 'MOTORCYCLIST'), ('ACCIDENT_TYPE', 'SPEED_ZONE',

In [14]:
def triple_screen_score(features, y, min_support=MIN_SUPPORT):
    a, b, c = features

    interaction = (
        screen_features_df[a].astype(str)
        + " | " + screen_features_df[b].astype(str)
        + " | " + screen_features_df[c].astype(str)
    )

    counts = interaction.value_counts()
    valid = counts[counts >= min_support].index
    interaction = interaction[interaction.isin(valid)]
    y_valid = y.loc[interaction.index]

    if interaction.empty:
        return None

    mi = mutual_info_score(interaction, y_valid)

    return {
        "feature_a": a,
        "feature_b": b,
        "feature_c": c,
        "mutual_information": float(mi),
        "n_valid_combinations": int(interaction.nunique()),
        "min_group_support": int(interaction.value_counts().min())
    }

triple_results = []

for triple in unique_triples:
    score = triple_screen_score(triple, screen_df["SEVERITY"])
    if score is not None:
        triple_results.append(score)

triple_results = pd.DataFrame(triple_results)

if not triple_results.empty:
    triple_results = triple_results.sort_values(
        "mutual_information", ascending=False
    ).reset_index(drop=True)

display(triple_results.head(15))


,feature_a,feature_b,feature_c,mutual_information,n_valid_combinations,min_group_support
0,DCA_CODE_DESCRIPTION,POLICE_ATTEND,MOTORCYCLIST,0.069892,161,101
1,DCA_CODE_DESCRIPTION,POLICE_ATTEND,MOTORCYCLE,0.069843,161,101
2,DCA_CODE_DESCRIPTION,POLICE_ATTEND,SPEED_ZONE,0.068853,268,100
3,DCA_CODE_DESCRIPTION,POLICE_ATTEND,DEG_URBAN_NAME,0.067187,199,100
4,DCA_CODE_DESCRIPTION,POLICE_ATTEND,OLD_DRIVER_75_AND_OVER,0.063343,140,100
5,DCA_CODE_DESCRIPTION,POLICE_ATTEND,YOUNG_DRIVER_18_25,0.063243,178,100
6,DRIVER,POLICE_ATTEND,SPEED_ZONE,0.063116,52,100
7,DCA_CODE_DESCRIPTION,POLICE_ATTEND,OLD_PED_65_AND_OVER,0.063007,127,102
8,DCA_CODE_DESCRIPTION,POLICE_ATTEND,ROAD_ROUTE_1,0.062983,218,101
9,DRIVER,POLICE_ATTEND,DEG_URBAN_NAME,0.062442,41,102


## 12. Final interaction candidates

The final output contains:

- the strongest two-way interactions,
- a small number of promising three-way interactions.

These are **candidate engineered features**, not final model features.

The next notebook will create these interactions and test whether they actually improve the severity models.


In [15]:
# Final two-way and three-way candidate tables

final_two_way = shortlist.copy()

final_three_way = (
    triple_results.head(5).copy()
    if not triple_results.empty
    else pd.DataFrame()
)

print("FINAL TWO-WAY CANDIDATES")
display(
    final_two_way[
        [
            "feature_a", "feature_b",
            "mutual_information",
            "incremental_mi",
            "fatal_range",
            "serious_range"
        ]
    ]
)

print("\nFINAL THREE-WAY CANDIDATES")
display(final_three_way)


FINAL TWO-WAY CANDIDATES


,feature_a,feature_b,mutual_information,incremental_mi,fatal_range,serious_range
0,DCA_CODE_DESCRIPTION,POLICE_ATTEND,0.061809,0.021305,0.118852,0.501038
1,ACCIDENT_TYPE,SPEED_ZONE,0.027637,0.014175,0.250000,0.425674
2,DCA_CODE_DESCRIPTION,SPEED_ZONE,0.043215,0.014045,0.186603,0.578442
3,SPEED_ZONE,DRIVER,0.027106,0.013877,0.065949,0.380930
4,PASSENGERVEHICLE,POLICE_ATTEND,0.054319,0.013815,0.032444,0.409254
5,DCA_CODE,SPEED_ZONE,0.027510,0.013672,0.233884,0.390515
6,DRIVER,POLICE_ATTEND,0.053952,0.013448,0.030146,0.414208
7,ACCIDENT_TYPE,POLICE_ATTEND,0.053544,0.013040,0.043984,0.419046
8,SPEED_ZONE,PASSENGERVEHICLE,0.025529,0.012300,0.053334,0.361670
9,DCA_CODE,POLICE_ATTEND,0.052418,0.011914,0.035892,0.406046



FINAL THREE-WAY CANDIDATES


,feature_a,feature_b,feature_c,mutual_information,n_valid_combinations,min_group_support
0,DCA_CODE_DESCRIPTION,POLICE_ATTEND,MOTORCYCLIST,0.069892,161,101
1,DCA_CODE_DESCRIPTION,POLICE_ATTEND,MOTORCYCLE,0.069843,161,101
2,DCA_CODE_DESCRIPTION,POLICE_ATTEND,SPEED_ZONE,0.068853,268,100
3,DCA_CODE_DESCRIPTION,POLICE_ATTEND,DEG_URBAN_NAME,0.067187,199,100
4,DCA_CODE_DESCRIPTION,POLICE_ATTEND,OLD_DRIVER_75_AND_OVER,0.063343,140,100


## 13. Save screening results

The saved CSV files allow Notebook 11 to consume the screening results without repeating the analysis.


In [16]:
# Save outputs
os.makedirs(DATA_DIR, exist_ok=True)

interaction_results.to_csv(
    os.path.join(DATA_DIR, "feature_interaction_screening_results.csv"),
    index=False
)

shortlist.to_csv(
    os.path.join(DATA_DIR, "feature_interaction_shortlist.csv"),
    index=False
)

if not triple_results.empty:
    triple_results.to_csv(
        os.path.join(DATA_DIR, "feature_interaction_triple_screening.csv"),
        index=False
    )

print("Saved:")
print("- feature_interaction_screening_results.csv")
print("- feature_interaction_shortlist.csv")
if not triple_results.empty:
    print("- feature_interaction_triple_screening.csv")


Saved:
- feature_interaction_screening_results.csv
- feature_interaction_shortlist.csv
- feature_interaction_triple_screening.csv


# Conclusion

This notebook does **not** assume that a particular interaction is useful.

Instead:

1. Candidate interactions were generated from the selected feature set and feature families.
2. Screening used the training set only.
3. Mutual information and severity-composition variation were used to identify promising combinations.
4. Incremental information was examined to distinguish interactions from simply strong individual features.
5. A small number of three-way combinations were explored conservatively.
6. The resulting shortlist will be tested as actual engineered features in the next notebook.

### Important interpretation

A high interaction-screening score does **not** prove that the feature improves predictive performance.

The final decision will be based on the downstream model comparison using the same evaluation framework already used in the project.
